In [1]:
import numpy as np
import pandas as pd
from numpy import loadtxt
from numpy import exp


from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import accuracy_score,mean_absolute_error,mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from keras.wrappers.scikit_learn import KerasRegressor,KerasClassifier
from torch.autograd import Variable

import torch
from torch import nn
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.data import Dataset, TensorDataset
import torch.nn.functional as F
#from pytorchtools import EarlyStopping


%matplotlib inline

pd.set_option('max_rows',999)
#Detect device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

### Import data

In [2]:
df_sunspots = pd.read_csv('data/data.csv.txt')

In [3]:
m = 10

# Split month
df_sunspots.columns = ['Data','Sunspots']
df_sunspots['Month'] = df_sunspots.Data.apply(lambda x: int(x.split('-')[1]))


### Calendar Features

In [4]:
df_calendar = pd.concat([pd.get_dummies(df_sunspots.iloc[m:].Month,prefix='Month_').reset_index(drop=True),df_sunspots.iloc[m:].reset_index(drop=True)],axis=1)
df_calendar.drop(columns=['Month','Data'],inplace=True)
df_calendar

,Month__1,Month__2,Month__3,Month__4,Month__5,Month__6,Month__7,Month__8,Month__9,Month__10,Month__11,Month__12,Sunspots
0,0,0,0,0,0,0,0,0,0,0,1,0,158.6
1,0,0,0,0,0,0,0,0,0,0,0,1,85.2
2,1,0,0,0,0,0,0,0,0,0,0,0,73.3
3,0,1,0,0,0,0,0,0,0,0,0,0,75.9
4,0,0,1,0,0,0,0,0,0,0,0,0,89.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2805,0,0,0,0,0,0,0,1,0,0,0,0,71.8
2806,0,0,0,0,0,0,0,0,1,0,0,0,50.3
2807,0,0,0,0,0,0,0,0,0,1,0,0,55.8
2808,0,0,0,0,0,0,0,0,0,0,1,0,33.3


### Embedding phase space

In [5]:
#build phase space
m = 10
df_phase=pd.DataFrame([df_sunspots.Sunspots.iloc[-m+d:d+1].values for d in range(df_sunspots.shape[0])]).iloc[m:]
df_phase.reset_index(drop=True,inplace=True)

df_phase.columns =['Sunspots_'+str(col-10) for col in df_phase.columns]
df_phase

,Sunspots_-10,Sunspots_-9,Sunspots_-8,Sunspots_-7,Sunspots_-6,Sunspots_-5,Sunspots_-4,Sunspots_-3,Sunspots_-2,Sunspots_-1,Sunspots_0
0,58.0,62.6,70.0,55.7,85.0,83.5,94.8,66.3,75.9,75.5,158.6
1,62.6,70.0,55.7,85.0,83.5,94.8,66.3,75.9,75.5,158.6,85.2
2,70.0,55.7,85.0,83.5,94.8,66.3,75.9,75.5,158.6,85.2,73.3
3,55.7,85.0,83.5,94.8,66.3,75.9,75.5,158.6,85.2,73.3,75.9
4,85.0,83.5,94.8,66.3,75.9,75.5,158.6,85.2,73.3,75.9,89.2
...,...,...,...,...,...,...,...,...,...,...,...
2805,94.7,98.1,127.0,84.3,51.0,66.5,80.7,99.2,91.1,82.2,71.8
2806,98.1,127.0,84.3,51.0,66.5,80.7,99.2,91.1,82.2,71.8,50.3
2807,127.0,84.3,51.0,66.5,80.7,99.2,91.1,82.2,71.8,50.3,55.8
2808,84.3,51.0,66.5,80.7,99.2,91.1,82.2,71.8,50.3,55.8,33.3


### dataprep complete

# Split month
df_sunspots = pd.read_csv('data/data.csv.txt')
df_sunspots.columns = ['Data','Sunspots']
df_sunspots['Month'] = df_sunspots.Data.apply(lambda x: int(x.split('-')[1]))

#build phase space
m = 10
df_phase=pd.DataFrame([df_sunspots.Sunspots.iloc[-m+d:d].values for d in range(df_sunspots.shape[0])])
df_phase.columns =['Sunspots_'+str(col-10) for col in df_phase.columns]
df_target = df_sunspots['Sunspots'].iloc[m:]

#get month dummies 
df_data=pd.concat([pd.get_dummies(df_sunspots.Month,prefix='Month_'),df_phase],axis=1).iloc[m:]
df_data.set_index(df_sunspots['Data'].iloc[m:],inplace=True)

### define Functions

In [87]:
def train_test_validation_tscv(df_data=pd.DataFrame(),
                               df_target=pd.DataFrame(),
                               tscv_train = TimeSeriesSplit(n_splits=2,test_size=30),
                               tscv_test = TimeSeriesSplit(n_splits=2,test_size=30)):
    X=df_data.values
    y=df_target.values
    for train_index, test_index in tscv_test.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
    for train_index, test_index in tscv_train.split(X_train):
        X_train_2, X_valid = X_train[train_index], X_train[test_index]
        y_train_2, y_valid = y_train[train_index], y_train[test_index]
        
    x_train_tensor = torch.from_numpy(X_train_2).float().to(device)
    y_train_tensor = torch.from_numpy(y_train_2).float().to(device)

    x_val_tensor = torch.from_numpy(X_valid).float().to(device)
    y_val_tensor = torch.from_numpy(y_valid).float().to(device)

    train_data = TensorDataset(x_train_tensor, y_train_tensor)
    train_loader = DataLoader(dataset=train_data, batch_size=10)

    val_data = TensorDataset(x_val_tensor, y_val_tensor)
    val_loader = DataLoader(dataset=val_data, batch_size=20)   
        
        
    return train_loader,val_loader, X_test,  y_test

def exp_decay(self, epoch):
    initial_lrate = 0.001
    k = 0.1
    lrate = initial_lrate * exp(-k*epoch)
    return lrate

class MLP(nn.Module):
    def __init__(self,dim_input=22):
        super().__init__()
        
        # define an MLP layers
        self.fc1 = nn.Linear(in_features=dim_input, out_features=32)
        self.fc2 = nn.Linear(in_features=32, out_features=8)
        self.output = nn.Linear(in_features=8, out_features=1)

    def forward(self, x):
        
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.output(x)
        return x
    
class MLP_combinado(nn.Module):
    def __init__(self,weights_fc1,layers,neurons, batch,p_drop):
        super().__init__()
        self.input_dim = weights_fc1.shape[1]
        self.weights_fc1 = weights_fc1
        self.layers = layers
        self.neurons = neurons
        self.batch= batch
        self.p_drop = p_drop
        
        # define an MLP layers
        self.fc1 = nn.Linear(in_features=self.input_dim, out_features=len(self.weights_fc1))
        self.fc1.weight = nn.Parameter(self.weights_fc1)
        self.batch1 = nn.BatchNorm1d(num_features=len(self.weights_fc1))
        self.dropout = nn.Dropout(self.p_drop)
        
        if layers>0:
            self.fcs = nn.ModuleList([nn.Linear(len(self.weights_fc1), self.neurons) if i==0 else nn.Linear(self.neurons, self.neurons) for i in range(self.layers)])

            if self.batch:   
                   self.batchs = nn.ModuleList([nn.BatchNorm1d(num_features=self.neurons) for i in range(self.layers)])

            self.fc2 = nn.Linear(in_features=self.neurons, out_features=8)
        else:
            self.fc2 = nn.Linear(in_features=len(self.weights_fc1), out_features=8)
        if self.batch:    
            self.batch2 = nn.BatchNorm1d(num_features=8)        
        
        self.output = nn.Linear(in_features=8, out_features=1)

    def forward(self, x):

        x = self.fc1(x)
        if self.batch:
            x = self.batch1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        
        for i in range(self.layers):
            x = self.fcs[i](x)
            if self.batch:
                x =self.batchs[i](x)
            x = torch.relu(x)
            x = self.dropout(x)
            
        x = self.fc2(x)
        if self.batch:
            x = self.batch2(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.output(x)
        return x
    
    
def train_model(df_data,df_target,model):

    train_loader,val_loader, X_test,  y_test = train_test_validation_tscv(df_data,df_target)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)
    epochs = 300

    # to track the training loss as the model trains
    train_losses = []
    # to track the validation loss as the model trains
    valid_losses = []
    # to track the average training loss per epoch as the model trains
    avg_train_losses = [0]
    # to track the average validation loss per epoch as the model trains
    avg_valid_losses = [0] 
    
    
    patience = 10
    trigger_times = 0
    
    for i in range(epochs):
        model.train()
        for x_batch, y_batch in train_loader:

            x_batch,y_batch = x_batch.to(device), y_batch.to(device) 
            
            # clear the gradients of all optimized variables
            optimizer.zero_grad()
            # Makes predictions
            yhat = model.forward(x_batch)
            # Computes loss
            loss = criterion(y_batch, yhat)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
        #
        scheduler.step()
        
           
        model.eval()      
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val = x_val
                y_val = y_val

                yhat = model.forward(x_val)
                #print("Predição:",yhat)
                val_loss = criterion(y_val, yhat)
                valid_losses.append(val_loss.item())

        train_loss = np.average(train_losses)
        valid_loss = np.average(valid_losses)
        avg_train_losses.append(train_loss)
        avg_valid_losses.append(valid_loss)        
        # clear lists to track next epoch
        train_losses = []
        valid_losses = []
        
        # early_stopping needs the validation loss to check if it has decresed, 
        # and if it has, it will make a checkpoint of the current model
        print('Epoch: %.2f Loss: %.2f - Loss_vald: %.2f' % (i,avg_train_losses[-1],avg_valid_losses[-1]) )   
        if avg_valid_losses[-1]  >= avg_valid_losses[-2]:
            trigger_times += 1
            print('trigger times:', trigger_times)

            if trigger_times >= patience:
                print('Early stopping!\nStart to test process.')
                break

        else:
            print('trigger times: 0')
            trigger_times = 0
            
         
            
    return model,avg_train_losses[1:], avg_valid_losses[1:], X_test, y_test

def evaluate(model, X_test,y_test):
    preds = []
    X_test_tensor = torch.from_numpy(X_test).float().to(device)
    model.eval()   
    with torch.no_grad():
        
        for val in X_test_tensor:
            y_hat = model.forward(val.unsqueeze(0))
            #print(y_hat)
            preds.append(y_hat.item())
    mae=mean_absolute_error(y_test,preds)
    print('MAE: %.2f' % mae)        
    return preds,mae

## Train model

In [33]:
model_calendar,losses_calendar, val_losses_calendar, X_calendar_test, y_calendar_test = train_model(df_calendar.drop(['Sunspots'],axis=1),
                                                                                                    df_calendar['Sunspots'], 
                                                                                                    MLP(12).to(device))
pred_calendar=evaluate(model_calendar,
         X_calendar_test,
         y_calendar_test)

C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([10, 1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([20, 1])) that is different to the input size (torch.Size([20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch: 0.00 Loss: 2270.56 - Loss_vald: 6731.49
trigger times: 1
Epoch: 1.00 Loss: 1880.99 - Loss_vald: 6430.45
trigger times: 0
Epoch: 2.00 Loss: 1875.76 - Loss_vald: 6479.06
trigger times: 1
Epoch: 3.00 Loss: 1869.89 - Loss_vald: 6629.76
trigger times: 2
Epoch: 4.00 Loss: 1864.21 - Loss_vald: 6840.74
trigger times: 3
Epoch: 5.00 Loss: 1858.84 - Loss_vald: 7088.05
trigger times: 4
Epoch: 6.00 Loss: 1853.83 - Loss_vald: 7354.26
trigger times: 5
Epoch: 7.00 Loss: 1849.15 - Loss_vald: 7626.50
trigger times: 6
Epoch: 8.00 Loss: 1844.75 - Loss_vald: 7894.64
trigger times: 7
Epoch: 9.00 Loss: 1840.55 - Loss_vald: 8151.27
trigger times: 8
Epoch: 10.00 Loss: 1836.48 - Loss_vald: 8390.55
trigger times: 9
Epoch: 11.00 Loss: 1832.50 - Loss_vald: 8609.51
trigger times: 10
Epoch: 12.00 Loss: 1828.58 - Loss_vald: 8806.48
trigger times: 11
Epoch: 13.00 Loss: 1824.74 - Loss_vald: 8981.32
trigger times: 12
Epoch: 14.00 Loss: 1820.97 - Loss_vald: 9134.86
trigger times: 13
Epoch: 15.00 Loss: 1817.32 - Lo

In [37]:
model_phase,losses_phase, val_losses_phase, X_phase_test, y_phase_test= train_model(df_phase.drop(['Sunspots_0'],axis=1),
                                                                                    df_phase['Sunspots_0'],
                                                                                    MLP(10).to(device))
preds_phase = evaluate(model_phase,X_phase_test,y_phase_test)

C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([10, 1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([20, 1])) that is different to the input size (torch.Size([20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch: 0.00 Loss: 625.44 - Loss_vald: 4816.20
trigger times: 1
Epoch: 1.00 Loss: 394.91 - Loss_vald: 2168.96
trigger times: 0
Epoch: 2.00 Loss: 378.16 - Loss_vald: 2707.74
trigger times: 1
Epoch: 3.00 Loss: 369.13 - Loss_vald: 1845.30
trigger times: 0
Epoch: 4.00 Loss: 354.40 - Loss_vald: 1331.12
trigger times: 0
Epoch: 5.00 Loss: 348.69 - Loss_vald: 1054.20
trigger times: 0
Epoch: 6.00 Loss: 342.81 - Loss_vald: 905.32
trigger times: 0
Epoch: 7.00 Loss: 339.65 - Loss_vald: 780.81
trigger times: 0
Epoch: 8.00 Loss: 337.30 - Loss_vald: 724.13
trigger times: 0
Epoch: 9.00 Loss: 335.24 - Loss_vald: 702.43
trigger times: 0
Epoch: 10.00 Loss: 333.96 - Loss_vald: 708.01
trigger times: 1
Epoch: 11.00 Loss: 332.59 - Loss_vald: 717.00
trigger times: 2
Epoch: 12.00 Loss: 331.01 - Loss_vald: 724.31
trigger times: 3
Epoch: 13.00 Loss: 330.13 - Loss_vald: 725.33
trigger times: 4
Epoch: 14.00 Loss: 328.96 - Loss_vald: 727.52
trigger times: 5
Epoch: 15.00 Loss: 327.90 - Loss_vald: 729.39
trigger times

[115.7938003540039,
 124.11936950683594,
 131.06597900390625,
 136.47789001464844,
 144.42526245117188,
 136.26539611816406,
 140.44781494140625,
 127.15556335449219,
 141.16004943847656,
 144.85092163085938,
 130.39981079101562,
 115.58235931396484,
 112.28710174560547,
 109.71778869628906,
 105.9737777709961,
 106.48941040039062,
 101.65621948242188,
 96.65821838378906,
 102.48942565917969,
 94.39729309082031,
 83.3999252319336,
 78.5954360961914,
 79.02275085449219,
 82.4640121459961,
 79.97650146484375,
 79.10425567626953,
 75.88088989257812,
 65.35445404052734,
 64.25939178466797,
 57.22028732299805]

In [38]:
fc1_weights = torch.cat([model_calendar.state_dict()['fc1.weight'],model_phase.state_dict()['fc1.weight']],axis=1)

In [57]:
model_combinado = MLP_combinado(fc1_weights,2,16,False,.25).to(device)

model_combinado,losses_phase, val_losses_phase, X_phase_test, y_phase_test= train_model(pd.concat([df_calendar.drop('Sunspots',axis=1),df_phase.drop('Sunspots_0',axis=1)],axis=1),
                                                                                    df_phase['Sunspots_0'],
                                                                                    model_combinado.to(device))
preds_comb =evaluate(model_combinado,X_phase_test,y_phase_test)

Epoch: 0.00 Loss: 1228.47 - Loss_vald: 2814.14
trigger times: 1
Epoch: 1.00 Loss: 1190.57 - Loss_vald: 11740.28
trigger times: 2
Epoch: 2.00 Loss: 989.09 - Loss_vald: 7639.07
trigger times: 0
Epoch: 3.00 Loss: 890.34 - Loss_vald: 4136.71
trigger times: 0
Epoch: 4.00 Loss: 816.29 - Loss_vald: 4018.86
trigger times: 0
Epoch: 5.00 Loss: 847.79 - Loss_vald: 7564.15
trigger times: 1
Epoch: 6.00 Loss: 853.27 - Loss_vald: 13435.36
trigger times: 2
Epoch: 7.00 Loss: 958.23 - Loss_vald: 15112.49
trigger times: 3
Epoch: 8.00 Loss: 925.68 - Loss_vald: 5572.43
trigger times: 0
Epoch: 9.00 Loss: 784.28 - Loss_vald: 8239.90
trigger times: 1
Epoch: 10.00 Loss: 735.50 - Loss_vald: 4523.76
trigger times: 0
Epoch: 11.00 Loss: 775.86 - Loss_vald: 4878.68
trigger times: 1
Epoch: 12.00 Loss: 768.20 - Loss_vald: 9921.82
trigger times: 2
Epoch: 13.00 Loss: 791.39 - Loss_vald: 5762.46
trigger times: 0
Epoch: 14.00 Loss: 709.08 - Loss_vald: 5637.29
trigger times: 0
Epoch: 15.00 Loss: 720.65 - Loss_vald: 5057.3

Epoch: 128.00 Loss: 646.10 - Loss_vald: 5055.28
trigger times: 0
Epoch: 129.00 Loss: 653.65 - Loss_vald: 5055.27
trigger times: 0
Epoch: 130.00 Loss: 660.42 - Loss_vald: 5055.22
trigger times: 0
Epoch: 131.00 Loss: 637.92 - Loss_vald: 5055.22
trigger times: 0
Epoch: 132.00 Loss: 626.83 - Loss_vald: 5055.21
trigger times: 0
Epoch: 133.00 Loss: 679.04 - Loss_vald: 5055.17
trigger times: 0
Epoch: 134.00 Loss: 674.92 - Loss_vald: 5055.15
trigger times: 0
Epoch: 135.00 Loss: 657.22 - Loss_vald: 5055.14
trigger times: 0
Epoch: 136.00 Loss: 661.20 - Loss_vald: 5055.13
trigger times: 0
Epoch: 137.00 Loss: 617.52 - Loss_vald: 5055.12
trigger times: 0
Epoch: 138.00 Loss: 650.84 - Loss_vald: 5055.12
trigger times: 0
Epoch: 139.00 Loss: 674.73 - Loss_vald: 5055.11
trigger times: 0
Epoch: 140.00 Loss: 665.05 - Loss_vald: 5055.11
trigger times: 0
Epoch: 141.00 Loss: 710.18 - Loss_vald: 5055.11
trigger times: 0
Epoch: 142.00 Loss: 661.18 - Loss_vald: 5055.11
trigger times: 0
Epoch: 143.00 Loss: 656.8

### Randon Search

In [101]:
n_estimators = 10
dict_param ={'layers':[0,1,2],
             'neurons':[16,32,64],
             'batch':[True,False],
             'p_drop':[0.15,0.25]}
grid = []
for key in dict_param.keys():
    values=[]
    
    indexs= np.random.randint(len(dict_param[key]),size=n_estimators)
    for index in indexs:
        values.append(dict_param[key][index])
    grid.append(values)
grid

[[1, 1, 1, 2, 0, 1, 2, 0, 2, 1],
 [64, 16, 64, 32, 16, 16, 16, 16, 64, 64],
 [False, True, True, True, False, False, True, True, False, False],
 [0.15, 0.25, 0.15, 0.25, 0.15, 0.25, 0.15, 0.15, 0.25, 0.25]]

#### grid best model

In [102]:
model_calendar=MLP(12).to(device)
model_calendar,losses_calendar, val_losses_calendar, X_calendar_test, y_calendar_test = train_model(df_calendar.drop(['Sunspots'],axis=1),
                                                                                                    df_calendar['Sunspots'], 
                                                                                                    model_calendar)
pred_calendar=evaluate(model_calendar,
         X_calendar_test,
         y_calendar_test)

model_phase = MLP(10).to(device)
model_phase,losses_phase, val_losses_phase, X_phase_test, y_phase_test= train_model(df_phase.drop(['Sunspots_0'],axis=1),
                                                                                    df_phase['Sunspots_0'],
                                                                                    model_phase)
pred_phase=evaluate(model_phase,X_phase_test,y_phase_test)
df_results = pd.DataFrame(columns=['model','loss','loss_val','test','layers','neurons','batch','p_prob'])
#concat both models weights
fc1_weights = torch.cat([model_calendar.state_dict()['fc1.weight'],model_phase.state_dict()['fc1.weight']],axis=1)

for i in range(n_estimators):
    # define model
    print('=======>>>>>>> Model' + str(i) + '<<<<<<<========')
    model_combinado = MLP_combinado(fc1_weights,grid[0][i],grid[1][i],grid[2][i],grid[3][i]).to(device)

    #train model
    model_combinado,losses_comb, val_losses_comb, X_phase_test, y_phase_test= train_model(pd.concat([df_calendar.drop('Sunspots',axis=1),df_phase.drop('Sunspots_0',axis=1)],axis=1),
                                                                                        df_phase['Sunspots_0'],
                                                                                        model_combinado.to(device))
    #evaluate model
    preds_comb,mae=evaluate(model_combinado,X_phase_test,y_phase_test)
    df_results = df_results.append({'model':model_combinado,
                   'loss':np.mean(losses_comb),
                   'loss_val':np.mean(val_losses_comb),
                   'loss_test':mae,
                   'layers': grid[0][i],
                   'neurons': grid[1][i],
                   'batch': grid[2][i],
                   'p_prob':grid[3][i]},ignore_index=True)

C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([10, 1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([20, 1])) that is different to the input size (torch.Size([20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch: 0.00 Loss: 2206.70 - Loss_vald: 7424.48
trigger times: 1
Epoch: 1.00 Loss: 1888.85 - Loss_vald: 6746.84
trigger times: 0
Epoch: 2.00 Loss: 1885.57 - Loss_vald: 6510.83
trigger times: 0
Epoch: 3.00 Loss: 1880.59 - Loss_vald: 6451.10
trigger times: 0
Epoch: 4.00 Loss: 1875.06 - Loss_vald: 6511.05
trigger times: 1
Epoch: 5.00 Loss: 1869.37 - Loss_vald: 6658.72
trigger times: 2
Epoch: 6.00 Loss: 1863.78 - Loss_vald: 6866.11
trigger times: 3
Epoch: 7.00 Loss: 1858.46 - Loss_vald: 7112.15
trigger times: 4
Epoch: 8.00 Loss: 1853.44 - Loss_vald: 7380.59
trigger times: 5
Epoch: 9.00 Loss: 1848.73 - Loss_vald: 7653.66
trigger times: 6
Epoch: 10.00 Loss: 1844.32 - Loss_vald: 7923.63
trigger times: 7
Epoch: 11.00 Loss: 1840.09 - Loss_vald: 8179.78
trigger times: 8
Epoch: 12.00 Loss: 1836.02 - Loss_vald: 8418.77
trigger times: 9
Epoch: 13.00 Loss: 1832.02 - Loss_vald: 8637.76
trigger times: 10
Early stopping!
Start to test process.
MAE: 50.53


C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([10, 1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([20, 1])) that is different to the input size (torch.Size([20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch: 0.00 Loss: 628.41 - Loss_vald: 3455.18
trigger times: 1
Epoch: 1.00 Loss: 391.04 - Loss_vald: 2649.57
trigger times: 0
Epoch: 2.00 Loss: 372.41 - Loss_vald: 2227.52
trigger times: 0
Epoch: 3.00 Loss: 363.52 - Loss_vald: 1750.75
trigger times: 0
Epoch: 4.00 Loss: 355.02 - Loss_vald: 1385.55
trigger times: 0
Epoch: 5.00 Loss: 348.98 - Loss_vald: 1122.95
trigger times: 0
Epoch: 6.00 Loss: 344.72 - Loss_vald: 889.83
trigger times: 0
Epoch: 7.00 Loss: 341.06 - Loss_vald: 873.90
trigger times: 0
Epoch: 8.00 Loss: 339.39 - Loss_vald: 784.23
trigger times: 0
Epoch: 9.00 Loss: 337.13 - Loss_vald: 730.01
trigger times: 0
Epoch: 10.00 Loss: 335.40 - Loss_vald: 710.02
trigger times: 0
Epoch: 11.00 Loss: 334.18 - Loss_vald: 710.36
trigger times: 1
Epoch: 12.00 Loss: 333.18 - Loss_vald: 710.79
trigger times: 2
Epoch: 13.00 Loss: 331.89 - Loss_vald: 715.31
trigger times: 3
Epoch: 14.00 Loss: 330.78 - Loss_vald: 719.30
trigger times: 4
Epoch: 15.00 Loss: 329.62 - Loss_vald: 721.54
trigger times

C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([10, 1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([20, 1])) that is different to the input size (torch.Size([20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch: 0.00 Loss: 1017.25 - Loss_vald: 965.76
trigger times: 1
Epoch: 1.00 Loss: 668.34 - Loss_vald: 1364.72
trigger times: 2
Epoch: 2.00 Loss: 792.07 - Loss_vald: 1511.79
trigger times: 3
Epoch: 3.00 Loss: 666.43 - Loss_vald: 1141.14
trigger times: 0
Epoch: 4.00 Loss: 565.12 - Loss_vald: 1788.34
trigger times: 1
Epoch: 5.00 Loss: 683.72 - Loss_vald: 729.31
trigger times: 0
Epoch: 6.00 Loss: 566.68 - Loss_vald: 2518.52
trigger times: 1
Epoch: 7.00 Loss: 587.53 - Loss_vald: 1199.22
trigger times: 0
Epoch: 8.00 Loss: 575.84 - Loss_vald: 5737.66
trigger times: 1
Epoch: 9.00 Loss: 548.74 - Loss_vald: 865.56
trigger times: 0
Epoch: 10.00 Loss: 567.63 - Loss_vald: 777.08
trigger times: 0
Epoch: 11.00 Loss: 546.38 - Loss_vald: 814.25
trigger times: 1
Epoch: 12.00 Loss: 509.72 - Loss_vald: 1237.02
trigger times: 2
Epoch: 13.00 Loss: 515.99 - Loss_vald: 1320.64
trigger times: 3
Epoch: 14.00 Loss: 541.14 - Loss_vald: 1116.15
trigger times: 0
Epoch: 15.00 Loss: 508.07 - Loss_vald: 1225.42
trigger

... (29 outputs left out)

Epoch: 128.00 Loss: 685.58 - Loss_vald: 4841.15
trigger times: 0
Epoch: 129.00 Loss: 725.38 - Loss_vald: 4841.05
trigger times: 0
Epoch: 130.00 Loss: 707.92 - Loss_vald: 4840.97
trigger times: 0
Epoch: 131.00 Loss: 720.03 - Loss_vald: 4840.93
trigger times: 0
Epoch: 132.00 Loss: 672.91 - Loss_vald: 4840.90
trigger times: 0
Epoch: 133.00 Loss: 685.20 - Loss_vald: 4840.85
trigger times: 0
Epoch: 134.00 Loss: 700.92 - Loss_vald: 4840.85
trigger times: 1
Epoch: 135.00 Loss: 713.87 - Loss_vald: 4840.80
trigger times: 0
Epoch: 136.00 Loss: 716.98 - Loss_vald: 4840.78
trigger times: 0
Epoch: 137.00 Loss: 758.84 - Loss_vald: 4840.75
trigger times: 0
Epoch: 138.00 Loss: 727.50 - Loss_vald: 4840.70
trigger times: 0
Epoch: 139.00 Loss: 703.74 - Loss_vald: 4840.69
trigger times: 0
Epoch: 140.00 Loss: 708.56 - Loss_vald: 4840.68
trigger times: 0
Epoch: 141.00 Loss: 709.43 - Loss_vald: 4840.66
trigger times: 0
Epoch: 142.00 Loss: 712.78 - Loss_vald: 4840.65
trigger times: 0
Epoch: 143.00 Loss: 725.6

C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([10, 1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\diego\anaconda3\lib\site-packages\torch\nn\modules\loss.py:529: UserWarning: Using a target size (torch.Size([20, 1])) that is different to the input size (torch.Size([20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch: 0.00 Loss: 1040.02 - Loss_vald: 1394.49
trigger times: 1
Epoch: 1.00 Loss: 1163.55 - Loss_vald: 2264.64
trigger times: 2
Epoch: 2.00 Loss: 890.74 - Loss_vald: 7004.76
trigger times: 3
Epoch: 3.00 Loss: 812.08 - Loss_vald: 1878.77
trigger times: 0
Epoch: 4.00 Loss: 875.34 - Loss_vald: 3942.86
trigger times: 1
Epoch: 5.00 Loss: 819.88 - Loss_vald: 4423.95
trigger times: 2
Epoch: 6.00 Loss: 748.74 - Loss_vald: 3259.67
trigger times: 0
Epoch: 7.00 Loss: 697.15 - Loss_vald: 7196.95
trigger times: 1
Epoch: 8.00 Loss: 673.86 - Loss_vald: 4456.16
trigger times: 0
Epoch: 9.00 Loss: 665.87 - Loss_vald: 2664.32
trigger times: 0
Epoch: 10.00 Loss: 729.14 - Loss_vald: 1681.77
trigger times: 0
Epoch: 11.00 Loss: 609.90 - Loss_vald: 2786.71
trigger times: 1
Epoch: 12.00 Loss: 637.65 - Loss_vald: 9035.49
trigger times: 2
Epoch: 13.00 Loss: 627.50 - Loss_vald: 1865.10
trigger times: 0
Epoch: 14.00 Loss: 697.73 - Loss_vald: 2178.51
trigger times: 1
Epoch: 15.00 Loss: 612.95 - Loss_vald: 2985.55
t

Epoch: 128.00 Loss: 579.41 - Loss_vald: 2805.27
trigger times: 0
Epoch: 129.00 Loss: 583.74 - Loss_vald: 2805.27
trigger times: 1
Epoch: 130.00 Loss: 577.32 - Loss_vald: 2805.22
trigger times: 0
Epoch: 131.00 Loss: 608.92 - Loss_vald: 2805.13
trigger times: 0
Epoch: 132.00 Loss: 580.66 - Loss_vald: 2805.12
trigger times: 0
Epoch: 133.00 Loss: 590.85 - Loss_vald: 2805.11
trigger times: 0
Epoch: 134.00 Loss: 580.65 - Loss_vald: 2805.09
trigger times: 0
Epoch: 135.00 Loss: 572.63 - Loss_vald: 2805.09
trigger times: 0
Epoch: 136.00 Loss: 577.36 - Loss_vald: 2805.05
trigger times: 0
Epoch: 137.00 Loss: 582.89 - Loss_vald: 2805.03
trigger times: 0
Epoch: 138.00 Loss: 579.89 - Loss_vald: 2805.01
trigger times: 0
Epoch: 139.00 Loss: 559.88 - Loss_vald: 2805.00
trigger times: 0
Epoch: 140.00 Loss: 569.11 - Loss_vald: 2804.99
trigger times: 0
Epoch: 141.00 Loss: 559.23 - Loss_vald: 2804.99
trigger times: 0
Epoch: 142.00 Loss: 576.68 - Loss_vald: 2804.98
trigger times: 0
Epoch: 143.00 Loss: 568.7

#### resume

In [105]:
df_results.sort_values('loss_test')

,model,loss,loss_val,test,layers,neurons,batch,p_prob,loss_test
4,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,0,16,False,0.15,19.503043
0,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,1,64,False,0.15,23.783862
3,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,2,32,True,0.25,31.652254
9,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,1,64,False,0.25,31.818659
5,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,1,16,False,0.25,33.256622
6,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,2,16,True,0.15,37.099038
7,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,0,16,True,0.15,37.667526
8,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,2,64,False,0.25,42.111866
2,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,1,64,True,0.15,49.936812
1,MLP_combinado(\n (fc1): Linear(in_features=22...,337.366448,922.574955,NaN,1,16,True,0.25,56.965274
